In [18]:
# Step 2: Import Python Packages: Pandas, Numpy, and Statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
import os

In [19]:
# Step 3: Set I/O Folders
InputPath = 'inputFolder'

os.makedirs('outputFolder', exist_ok=True)
OutputPath = 'outputFolder'

In [20]:
df = pd.read_csv(InputPath + '/CaseStudy_PostTrade.csv')
df.head()

,Broker,Trade Date,Symbol,ADV,Volatility,Beta,Mkt Cap (MM),Side,Order Shares,Executed Shares,POV,Pavg,Pd,P0,Pn,VWAP
0,Broker A,8/5/2024,AAPL,72106864.0,0.68,1.041596,2901664,Buy,2294800,2294800,0.153908,197.52,193.27,193.49,199.25,197.379224
1,Broker C,8/5/2024,MSFT,26193634.0,0.40,0.991805,2669692,Buy,658600,658600,0.151336,367.34,362.82,362.88,370.55,367.287875
2,Broker B,8/5/2024,NVDA,323825056.0,0.81,1.904370,2364604,Sell,14230300,14230300,0.148172,96.08,98.77,98.63,97.66,96.097986
3,Broker A,8/5/2024,AMZN,56974192.0,0.58,1.184176,1775661,Sell,519200,519200,0.091066,167.46,169.60,169.56,168.62,167.565463
4,Broker A,8/5/2024,META,20542520.0,0.67,1.181923,1223500,Sell,526900,526900,0.173758,481.83,491.33,491.27,489.22,481.921026


In [21]:
# Step 4: Set MI Parameters - Calculated from Non-Linear Regression Analysis
a1 = 883.58
a2 = 0.35
a3 = 0.75
a4 = 0.82
b1 = 0.96

In [22]:
df['Side'] = df['Side'].map({'Buy': 1, 'Sell': -1}) # Map side to value 1 or -1
df['Size_fraction'] = df['Order Shares'] / df['ADV']

## Implementation Shortfall

In [23]:
COMMISSION_PER_SHARE = 0.01

df['Fixed_Cost'] = df['Executed Shares'] * COMMISSION_PER_SHARE
df['Delay_Cost'] = df['Side'] * df['Order Shares'] * (df['P0'] - df['Pd'])
df['Execution_Cost'] = df['Side'] * df['Executed Shares'] * (df['Pavg'] - df['P0'])
df['Opportunity_Cost'] = df['Side'] * (df['Order Shares'] - df['Executed Shares']) * (df['Pn'] - df['P0'])
df['IS_Total'] = df['Delay_Cost'] + df['Execution_Cost'] + df['Opportunity_Cost'] + df['Fixed_Cost']

In [24]:
df.head()

,Broker,Trade Date,Symbol,ADV,Volatility,Beta,Mkt Cap (MM),Side,Order Shares,Executed Shares,...,Pd,P0,Pn,VWAP,Size_fraction,Fixed_Cost,Delay_Cost,Execution_Cost,Opportunity_Cost,IS_Total
0,Broker A,8/5/2024,AAPL,72106864.0,0.68,1.041596,2901664,1,2294800,2294800,...,193.27,193.49,199.25,197.379224,0.031825,22948.0,504856.0,9248044.0,0.0,9775848.0
1,Broker C,8/5/2024,MSFT,26193634.0,0.40,0.991805,2669692,1,658600,658600,...,362.82,362.88,370.55,367.287875,0.025144,6586.0,39516.0,2937356.0,0.0,2983458.0
2,Broker B,8/5/2024,NVDA,323825056.0,0.81,1.904370,2364604,-1,14230300,14230300,...,98.77,98.63,97.66,96.097986,0.043944,142303.0,1992242.0,36287265.0,-0.0,38421810.0
3,Broker A,8/5/2024,AMZN,56974192.0,0.58,1.184176,1775661,-1,519200,519200,...,169.60,169.56,168.62,167.565463,0.009113,5192.0,20768.0,1090320.0,-0.0,1116280.0
4,Broker A,8/5/2024,META,20542520.0,0.67,1.181923,1223500,-1,526900,526900,...,491.33,491.27,489.22,481.921026,0.025649,5269.0,31614.0,4973936.0,-0.0,5010819.0


## Arrival Cost

In [25]:
df['Arrival_Cost_bp'] = df['Side'] * ((df['Pavg'] - df['P0']) / df['P0']) * 10000

df.head()

,Broker,Trade Date,Symbol,ADV,Volatility,Beta,Mkt Cap (MM),Side,Order Shares,Executed Shares,...,P0,Pn,VWAP,Size_fraction,Fixed_Cost,Delay_Cost,Execution_Cost,Opportunity_Cost,IS_Total,Arrival_Cost_bp
0,Broker A,8/5/2024,AAPL,72106864.0,0.68,1.041596,2901664,1,2294800,2294800,...,193.49,199.25,197.379224,0.031825,22948.0,504856.0,9248044.0,0.0,9775848.0,208.279498
1,Broker C,8/5/2024,MSFT,26193634.0,0.40,0.991805,2669692,1,658600,658600,...,362.88,370.55,367.287875,0.025144,6586.0,39516.0,2937356.0,0.0,2983458.0,122.905644
2,Broker B,8/5/2024,NVDA,323825056.0,0.81,1.904370,2364604,-1,14230300,14230300,...,98.63,97.66,96.097986,0.043944,142303.0,1992242.0,36287265.0,-0.0,38421810.0,258.542026
3,Broker A,8/5/2024,AMZN,56974192.0,0.58,1.184176,1775661,-1,519200,519200,...,169.56,168.62,167.565463,0.009113,5192.0,20768.0,1090320.0,-0.0,1116280.0,123.849965
4,Broker A,8/5/2024,META,20542520.0,0.67,1.181923,1223500,-1,526900,526900,...,491.27,489.22,481.921026,0.025649,5269.0,31614.0,4973936.0,-0.0,5010819.0,192.155027


## VWAP Slippage

In [26]:
df['VWAP_Slippage_bp'] = df['Side'] * ((df['Pavg'] - df['VWAP']) / df['VWAP']) * 10000

df.head()

,Broker,Trade Date,Symbol,ADV,Volatility,Beta,Mkt Cap (MM),Side,Order Shares,Executed Shares,...,Pn,VWAP,Size_fraction,Fixed_Cost,Delay_Cost,Execution_Cost,Opportunity_Cost,IS_Total,Arrival_Cost_bp,VWAP_Slippage_bp
0,Broker A,8/5/2024,AAPL,72106864.0,0.68,1.041596,2901664,1,2294800,2294800,...,199.25,197.379224,0.031825,22948.0,504856.0,9248044.0,0.0,9775848.0,208.279498,7.132270
1,Broker C,8/5/2024,MSFT,26193634.0,0.40,0.991805,2669692,1,658600,658600,...,370.55,367.287875,0.025144,6586.0,39516.0,2937356.0,0.0,2983458.0,122.905644,1.419187
2,Broker B,8/5/2024,NVDA,323825056.0,0.81,1.904370,2364604,-1,14230300,14230300,...,97.66,96.097986,0.043944,142303.0,1992242.0,36287265.0,-0.0,38421810.0,258.542026,1.871679
3,Broker A,8/5/2024,AMZN,56974192.0,0.58,1.184176,1775661,-1,519200,519200,...,168.62,167.565463,0.009113,5192.0,20768.0,1090320.0,-0.0,1116280.0,123.849965,6.293863
4,Broker A,8/5/2024,META,20542520.0,0.67,1.181923,1223500,-1,526900,526900,...,489.22,481.921026,0.025649,5269.0,31614.0,4973936.0,-0.0,5010819.0,192.155027,1.888812


## Value-Add

In [ ]:
# ------------ Calculate MI, PA, TR ------------
df['I_star'] = a1 * (df['Size_fraction']**a2) * (df['Volatility'])
# Market Impact (MI) = I* * (b1 * POV^a4 + (1 - b1))
df['MI_bp'] = df['I_star'] * (b1 * (df['POV']**a4) + (1 - b1))
# Timing Risk (TR) = sigma * sqrt(1/3 * 1/250 * Size * (1-POV)/POV) * 10000
df['TR_bp'] = df['Volatility'] * np.sqrt((1/3) * (1/250) * df['Size_fraction'] * (1-df['POV'])/df['POV']) * 10000

df['Value_Add_bp'] = df['MI_bp'] - df['Arrival_Cost_bp']

df.head()

,Broker,Trade Date,Symbol,ADV,Volatility,Beta,Mkt Cap (MM),Side,Order Shares,Executed Shares,...,Delay_Cost,Execution_Cost,Opportunity_Cost,IS_Total,Arrival_Cost_bp,VWAP_Slippage_bp,I_star,MI_bp,TR_bp,Value_Add_bp
0,Broker A,8/5/2024,AAPL,72106864.0,0.68,1.041596,2901664,1,2294800,2294800,...,504856.0,9248044.0,0.0,9775848.0,208.279498,7.132270,179.772671,44.391493,103.858032,-163.888004
1,Broker C,8/5/2024,MSFT,26193634.0,0.40,0.991805,2669692,1,658600,658600,...,39516.0,2937356.0,0.0,2983458.0,122.905644,1.419187,97.376679,23.768719,54.845367,-99.136925
2,Broker B,8/5/2024,NVDA,323825056.0,0.81,1.904370,2364604,-1,14230300,14230300,...,1992242.0,36287265.0,-0.0,38421810.0,258.542026,1.871679,239.743675,57.678722,148.661965,-200.863304
3,Broker A,8/5/2024,AMZN,56974192.0,0.58,1.184176,1775661,-1,519200,519200,...,20768.0,1090320.0,-0.0,1116280.0,123.849965,6.293863,98.981413,17.279008,63.872550,-106.570956
4,Broker A,8/5/2024,META,20542520.0,0.67,1.181923,1223500,-1,526900,526900,...,31614.0,4973936.0,-0.0,5010819.0,192.155027,1.888812,164.246736,44.112342,85.440254,-148.042684


## Z-Score

In [28]:
df['Z_Score'] = df['Value_Add_bp'] / df['TR_bp']

df.head()

,Broker,Trade Date,Symbol,ADV,Volatility,Beta,Mkt Cap (MM),Side,Order Shares,Executed Shares,...,Execution_Cost,Opportunity_Cost,IS_Total,Arrival_Cost_bp,VWAP_Slippage_bp,I_star,MI_bp,TR_bp,Value_Add_bp,Z_Score
0,Broker A,8/5/2024,AAPL,72106864.0,0.68,1.041596,2901664,1,2294800,2294800,...,9248044.0,0.0,9775848.0,208.279498,7.132270,179.772671,44.391493,103.858032,-163.888004,-1.578000
1,Broker C,8/5/2024,MSFT,26193634.0,0.40,0.991805,2669692,1,658600,658600,...,2937356.0,0.0,2983458.0,122.905644,1.419187,97.376679,23.768719,54.845367,-99.136925,-1.807572
2,Broker B,8/5/2024,NVDA,323825056.0,0.81,1.904370,2364604,-1,14230300,14230300,...,36287265.0,-0.0,38421810.0,258.542026,1.871679,239.743675,57.678722,148.661965,-200.863304,-1.351141
3,Broker A,8/5/2024,AMZN,56974192.0,0.58,1.184176,1775661,-1,519200,519200,...,1090320.0,-0.0,1116280.0,123.849965,6.293863,98.981413,17.279008,63.872550,-106.570956,-1.668494
4,Broker A,8/5/2024,META,20542520.0,0.67,1.181923,1223500,-1,526900,526900,...,4973936.0,-0.0,5010819.0,192.155027,1.888812,164.246736,44.112342,85.440254,-148.042684,-1.732704


## Save Broker Report Card to CSV

In [29]:
def get_report_card(data):
    # Grouping results by Broker
    report = data.groupby('Broker').agg(
        Orders=('Order Shares', 'count'),
        Avg_Size=('Size_fraction', 'mean'),
        IS_Total=('IS_Total', 'sum'),
        Delay_Cost=('Delay_Cost', 'sum'),
        Execution_Cost=('Execution_Cost', 'sum'),
        Opportunity_Cost=('Opportunity_Cost', 'sum'),
        Fixed_Cost=('Fixed_Cost', 'sum'),
        Arrival_bp=('Arrival_Cost_bp', 'mean'),
        VWAP_Slip_bp=('VWAP_Slippage_bp', 'mean'),
        Value_Add_bp=('Value_Add_bp', 'mean'),
        Z_Score=('Z_Score', 'mean')
    )
    return report

# --- GENERATING THE TABLES ---

# All Orders
all_orders_rpt = get_report_card(df)

small_orders_rpt = get_report_card(df[df['Size_fraction'] <= 0.05])

large_orders_rpt = get_report_card(df[df['Size_fraction'] > 0.05])

# All Orders report to csv
all_orders_rpt.to_csv(OutputPath + '/all_orders_rpt.csv')

# Display All Orders Report
print(all_orders_rpt.round())


          Orders  Avg_Size      IS_Total  Delay_Cost  Execution_Cost  \
Broker                                                                 
Broker A    1005       0.0  6.991052e+08  23436598.0     659127413.0   
Broker B    1026       0.0  1.040730e+09  24877886.0     996399348.0   
Broker C     966       0.0  7.878193e+08  23372496.0     729022425.0   
Broker D    1001       0.0  6.390293e+08  27006695.0     585242196.0   
Broker E    1012       0.0  9.371193e+08  31465691.0     889197438.0   

          Opportunity_Cost  Fixed_Cost  Arrival_bp  VWAP_Slip_bp  \
Broker                                                             
Broker A        13128723.0   3412461.0       163.0          14.0   
Broker B        15343349.0   4109479.0       164.0           7.0   
Broker C        32288145.0   3136189.0       164.0           2.0   
Broker D        23999570.0   2780834.0       157.0           7.0   
Broker E        12716105.0   3740113.0       156.0          15.0   

          Value_Ad

## Answer to Questions

1. 

2. 